# AIC 2026 — Notebook 01: Build & Audit Indices (Local / Kaggle 2×T4)

Kế hoạch nguồn: `Kiet-Prompt/Prompt_Plan_Local.md`.

**Nhiệm vụ notebook này**
1. Tự dò dataset root trong `/kaggle/input` (có thể override bằng `CFG`).
2. Audit schema/coverage từng modality → `audit_report.csv`.
3. Sinh **canonical records** (keyframes, caption, OCR, object, transcript segment, summary/metadata).
4. Build & persist index: SigLIP2 FAISS (visual), BGE-M3 FAISS (caption / transcript-segment / summary-video), BM25 multi-field, object inverted index.
5. Integrity check (NaN/Inf, dim, row-count, mapping) + smoke query + `manifest.json`.

**Ràng buộc**: toàn bộ code nằm trong notebook (không tạo file `.py`). Artifact runtime ghi vào `/kaggle/working/artifacts`.
Mọi stage đều **checkpoint theo shard** và **skip nếu fingerprint không đổi** → chạy lại an toàn.

In [ ]:
# ============================================================
# CELL 1 — Environment probe (không cài gì nếu đã có)
# ============================================================
import sys, os, subprocess, importlib

def have(mod):
    try:
        importlib.import_module(mod); return True
    except Exception:
        return False

NEED = {"faiss": "faiss-cpu", "pyarrow": "pyarrow"}
missing = [pkg for mod, pkg in NEED.items() if not have(mod)]
if missing:
    print("pip install:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=False)

import numpy as np, pandas as pd, faiss, torch, platform
print("python  :", platform.python_version())
print("torch   :", torch.__version__, "| cuda:", torch.cuda.is_available(), "| n_gpu:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU{i}: {p.name} {p.total_memory/1e9:.1f} GB")
print("numpy   :", np.__version__, "| pandas:", pd.__version__)

In [ ]:
# ============================================================
# CELL 2 — CFG: config tập trung (chỉ sửa ở đây)
# ============================================================
import os, glob, json, hashlib, time, gc, re, pickle, unicodedata, math, traceback
from pathlib import Path

CFG = dict(
    # --- Nguồn dữ liệu: None = auto-discover trong SEARCH_ROOTS ---
    SEARCH_ROOTS   = ["/kaggle/input", "./Feature_Dataset", "."],
    DATASET_ROOT   = None,   # aic-dataset (Videos_*/ , Keyframes_*/)
    FEATURE_ROOT   = None,   # feature-aic-2026
    QUERY_ROOT     = None,   # dethithunghiem (query-*.txt)
    ART_DIR        = "/kaggle/working/artifacts",

    # --- Model ---
    BGE_M3         = "BAAI/bge-m3",
    BGE_M3_REV     = None,          # pin revision khi đã chốt
    BGE_DIM        = 1024,
    SIGLIP_DIM     = 1536,
    LOCAL_MODEL_DIR= None,          # /kaggle/input/<model-cache>  nếu offline

    # --- Build params ---
    LIMIT_VIDEOS   = None,          # int để smoke test nhanh (vd 20)
    BGE_BATCH      = 16,
    BGE_MAXLEN     = 512,
    BGE_FP16       = True,
    SHARD_SIZE     = 20000,         # số document / shard checkpoint
    TRANSCRIPT_CHUNK_SEC   = 30.0,  # gộp segment ASR thành chunk ~30s
    TRANSCRIPT_OVERLAP_SEC = 10.0,
    OBJ_CONF_MIN   = 0.30,
    OCR_CONF_MIN   = 0.35,
    SEED           = 20260824,
    WALL_BUDGET_H  = 10.5,          # early-stop trước timeout 12h
)

np.random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"])
T0 = time.time()

def budget_left():
    return CFG["WALL_BUDGET_H"] * 3600 - (time.time() - T0)

def check_budget(tag=""):
    if budget_left() <= 0:
        raise TimeoutError(f"Wall-clock budget exhausted at [{tag}] — Save Version rồi resume.")

class Timer:
    def __init__(self, tag): self.tag = tag
    def __enter__(self): self.t = time.time(); print(f"[{self.tag}] start"); return self
    def __exit__(self, *a):
        vr = ""
        if torch.cuda.is_available():
            vr = " | peakVRAM " + " ".join(f"g{i}={torch.cuda.max_memory_allocated(i)/1e9:.1f}G"
                                           for i in range(torch.cuda.device_count()))
        print(f"[{self.tag}] done in {time.time()-self.t:.1f}s{vr}")

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

print("CFG loaded. budget_left = %.2f h" % (budget_left()/3600))

In [ ]:
# ============================================================
# CELL 3 — Auto-discovery dataset root + modality paths
# ============================================================
def _find_dir(name_patterns, roots, must_contain=None, maxdepth=4):
    """Trả về path đầu tiên khớp bất kỳ pattern glob."""
    for root in roots:
        if not os.path.isdir(root):
            continue
        for pat in name_patterns:
            for d in range(1, maxdepth + 1):
                for p in sorted(glob.glob(os.path.join(root, *(["*"] * (d - 1)), pat))):
                    if not os.path.isdir(p):
                        continue
                    if must_contain and not any(glob.glob(os.path.join(p, m)) for m in must_contain):
                        continue
                    return p
    return None

ROOTS = [r for r in CFG["SEARCH_ROOTS"] if os.path.isdir(r)]
print("search roots:", ROOTS)

DATASET_ROOT = CFG["DATASET_ROOT"] or _find_dir(["aic-dataset", "aic_dataset", "*aic-dataset*"], ROOTS,
                                                must_contain=["Videos_*", "Keyframes_*"])
FEATURE_ROOT = CFG["FEATURE_ROOT"] or _find_dir(["feature-aic-2026", "*feature*aic*", "Feature_Dataset"], ROOTS,
                                                must_contain=["siglip2-features-giant-opt", "Image_captioning"])
QUERY_ROOT   = CFG["QUERY_ROOT"]   or _find_dir(["dethithunghiem", "*dethi*", "SOTUYEN1-bo-de-thi"], ROOTS,
                                                must_contain=["query-*.txt", "*/query-*.txt"])

if FEATURE_ROOT is None and os.path.isdir("./Feature_Dataset"):
    FEATURE_ROOT = "./Feature_Dataset"

PATHS = {}
def _sub(key, pats, must=None):
    base = [FEATURE_ROOT] if FEATURE_ROOT else []
    PATHS[key] = _find_dir(pats, base, must_contain=must, maxdepth=3)

_sub("siglip",     ["siglip2-features-giant-opt", "*siglip2*"], ["*.npy"])
_sub("clip",       ["clip-features-32", "clip-features-32-aic25-b1"], ["*.npy"])
_sub("mapkf",      ["map-keyframes", "map-keyframes-aic25-b1"], ["*.csv"])
_sub("media",      ["media-info", "media-info-aic25-b1"], ["*.json"])
_sub("objects",    ["objects", "objects-aic25-b1"], ["*/*.json"])
_sub("caption",    ["Image_captioning", "*caption*"], ["*.json"])
_sub("ocr",        ["OCR_EasyOCR_VietOCR", "OCR*", "*ocr*"], ["*.json"])
_sub("summary",    ["Summary_video", "*summary*"], ["*.json"])
_sub("asr_vi",     ["Transcript_Extract"], ["*/*/*.json", "*/*.json"])
_sub("asr_en",     ["Transcript_Translated"], ["*/*/*.json", "*/*.json"])

def fast_glob(roots, pat, maxdepth=5, limit=None):
    """Quét theo từng tầng bằng glob KHÔNG recursive — nhanh hơn nhiều so với
    glob(**, recursive=True) khi dataset Kaggle có hàng trăm nghìn file."""
    out = []
    for root in ([roots] if isinstance(roots, str) else roots):
        if not root or not os.path.isdir(root):
            continue
        for d in range(1, maxdepth + 1):
            out += glob.glob(os.path.join(root, *(["*"] * (d - 1)), pat))
            if limit and len(out) >= limit:
                return out[:limit]
    return out

VIDEO_GLOBS, KEYFRAME_DIRS = [], []
if DATASET_ROOT:
    VIDEO_GLOBS   = sorted(set(fast_glob(DATASET_ROOT, "*.mp4", maxdepth=4)))
    KEYFRAME_DIRS = sorted(glob.glob(os.path.join(DATASET_ROOT, "Keyframes_*", "keyframes")))
    if not KEYFRAME_DIRS:
        KEYFRAME_DIRS = sorted(set(fast_glob(DATASET_ROOT, "keyframes", maxdepth=3)))

ART = Path(CFG["ART_DIR"]); ART.mkdir(parents=True, exist_ok=True)
for sub in ("ckpt", "index", "records"):
    (ART / sub).mkdir(exist_ok=True)

print("DATASET_ROOT :", DATASET_ROOT)
print("FEATURE_ROOT :", FEATURE_ROOT)
print("QUERY_ROOT   :", QUERY_ROOT)
print("ART          :", ART)
print("videos found :", len(VIDEO_GLOBS), "| keyframe dirs:", len(KEYFRAME_DIRS))
for k, v in PATHS.items():
    print(f"  {k:9s}: {v}")
missing_mod = [k for k, v in PATHS.items() if v is None]
if missing_mod:
    print("!! modality không tìm thấy (fallback is_missing=True):", missing_mod)

In [ ]:
# ============================================================
# CELL 4 — Utils: IO, hash/fingerprint, checkpoint, text norm
# ============================================================
def jload(p, default=None):
    try:
        with open(p, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return default

def jdump(obj, p):
    p = str(p); os.makedirs(os.path.dirname(p), exist_ok=True)
    tmp = p + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=1)
    os.replace(tmp, p)

def sha(*parts, n=16):
    h = hashlib.sha256()
    for x in parts:
        h.update(json.dumps(x, sort_keys=True, default=str).encode("utf-8"))
    return h.hexdigest()[:n]

def dir_fingerprint(path, pattern="**/*", limit=None):
    """Fingerprint nội dung thư mục theo (relpath, size) — đủ để phát hiện thay đổi."""
    if not path or not os.path.isdir(path):
        return "absent"
    items = []
    for p in sorted(glob.glob(os.path.join(path, pattern), recursive=True))[:limit]:
        if os.path.isfile(p):
            items.append((os.path.relpath(p, path), os.stat(p).st_size))
    return sha(items, len(items))

def save_parquet(df, name):
    p = ART / "records" / f"{name}.parquet"
    df.to_parquet(p, index=False)
    print(f"  -> {name}.parquet  rows={len(df):,}  {p.stat().st_size/1e6:.1f} MB")
    return p

def load_parquet(name):
    p = ART / "records" / f"{name}.parquet"
    return pd.read_parquet(p) if p.exists() else None

STAGE_STATE_P = ART / "stage_state.json"
STAGE = jload(STAGE_STATE_P, {}) or {}

def stage_done(name, fp):
    return STAGE.get(name, {}).get("fp") == fp

def mark_stage(name, fp, **extra):
    STAGE[name] = dict(fp=fp, ts=time.strftime("%Y-%m-%d %H:%M:%S"), **extra)
    jdump(STAGE, STAGE_STATE_P)

ERRORS = []
def log_err(stage, key, exc):
    ERRORS.append(dict(stage=stage, key=str(key), err=repr(exc)[:400]))
    if len(ERRORS) <= 20:
        print(f"  !! {stage} / {key}: {repr(exc)[:200]}")

# --- Vietnamese text normalisation (cho BM25 multi-field) ---
_VN_MAP = str.maketrans({"đ": "d", "Đ": "D"})
def nfc(s):
    return unicodedata.normalize("NFC", s or "")
def strip_diacritics(s):
    s = unicodedata.normalize("NFD", (s or "").translate(_VN_MAP))
    return "".join(c for c in s if unicodedata.category(c) != "Mn")
TOKEN_RE = re.compile(r"[0-9]+(?:[.,:/][0-9]+)*|[^\W_]+", re.UNICODE)
def tokenize(s, fold=True):
    s = nfc(s).lower() if fold else nfc(s)
    return TOKEN_RE.findall(s)
def tokenize_nodia(s):
    return TOKEN_RE.findall(strip_diacritics(nfc(s)).lower())

print("utils ready | stage đã hoàn tất:", list(STAGE.keys()))

In [ ]:
# ============================================================
# CELL 5 — Video inventory + audit coverage từng modality
# ============================================================
def list_video_ids():
    ids = set()
    if PATHS["mapkf"]:
        ids |= {Path(p).stem for p in glob.glob(os.path.join(PATHS["mapkf"], "*.csv"))}
    for k in ("siglip", "caption", "ocr", "summary"):
        if PATHS[k]:
            ids |= {Path(p).stem for p in glob.glob(os.path.join(PATHS[k], "*.npy"))}
            ids |= {Path(p).stem for p in glob.glob(os.path.join(PATHS[k], "*.json"))}
    for p in VIDEO_GLOBS:
        ids.add(Path(p).stem)
    return sorted(i for i in ids if re.fullmatch(r"L\d+_V\d+", i))

VIDEO_IDS = list_video_ids()
if CFG["LIMIT_VIDEOS"]:
    VIDEO_IDS = VIDEO_IDS[: CFG["LIMIT_VIDEOS"]]
    print(f"** LIMIT_VIDEOS={CFG['LIMIT_VIDEOS']} → smoke mode")
print("total video_id:", len(VIDEO_IDS), VIDEO_IDS[:3], "...")

ASR_VI_INDEX, ASR_EN_INDEX = {}, {}
for key, store in (("asr_vi", ASR_VI_INDEX), ("asr_en", ASR_EN_INDEX)):
    if PATHS[key]:
        for p in fast_glob(PATHS[key], "*.json", maxdepth=4):
            store[Path(p).stem] = p
VIDEO_FILE = {Path(p).stem: p for p in VIDEO_GLOBS}
KF_DIR = {}
for d in KEYFRAME_DIRS:
    for sub in sorted(glob.glob(os.path.join(d, "*"))):
        if os.path.isdir(sub):
            KF_DIR[os.path.basename(sub)] = sub

def mod_path(kind, vid):
    if kind == "siglip":  return PATHS["siglip"] and os.path.join(PATHS["siglip"], f"{vid}.npy")
    if kind == "clip":    return PATHS["clip"] and os.path.join(PATHS["clip"], f"{vid}.npy")
    if kind == "mapkf":   return PATHS["mapkf"] and os.path.join(PATHS["mapkf"], f"{vid}.csv")
    if kind == "media":   return PATHS["media"] and os.path.join(PATHS["media"], f"{vid}.json")
    if kind == "caption": return PATHS["caption"] and os.path.join(PATHS["caption"], f"{vid}.json")
    if kind == "ocr":     return PATHS["ocr"] and os.path.join(PATHS["ocr"], f"{vid}.json")
    if kind == "summary": return PATHS["summary"] and os.path.join(PATHS["summary"], f"{vid}.json")
    if kind == "objects": return PATHS["objects"] and os.path.join(PATHS["objects"], vid)
    if kind == "asr_vi":  return ASR_VI_INDEX.get(vid)
    if kind == "asr_en":  return ASR_EN_INDEX.get(vid)
    if kind == "video":   return VIDEO_FILE.get(vid)
    if kind == "kf":      return KF_DIR.get(vid)
    return None

MODS = ["siglip", "clip", "mapkf", "media", "caption", "ocr", "summary",
        "objects", "asr_vi", "asr_en", "video", "kf"]

rows = []
for vid in VIDEO_IDS:
    r = {"video_id": vid}
    for m in MODS:
        p = mod_path(m, vid)
        r[m] = bool(p) and os.path.exists(p)
    rows.append(r)
audit = pd.DataFrame(rows)
print("\n=== COVERAGE (%) ===")
print((audit[MODS].mean() * 100).round(1).to_string())
audit.to_csv(ART / "audit_coverage.csv", index=False)
print("\nthiếu siglip:", audit.loc[~audit.siglip, "video_id"].tolist()[:10])
print("thiếu ocr   :", audit.loc[~audit.ocr, "video_id"].tolist()[:10])
print("thiếu video :", audit.loc[~audit.video, "video_id"].tolist()[:10])

In [ ]:
# ============================================================
# CELL 6 — Audit schema THỰC TẾ (không tin README) + P0 assertions
# ============================================================
def audit_schema(sample_n=8):
    rep = []
    for vid in VIDEO_IDS[:sample_n]:
        rec = {"video_id": vid}
        mp = mod_path("mapkf", vid)
        if mp and os.path.exists(mp):
            df = pd.read_csv(mp)
            rec["mapkf_cols"] = ",".join(df.columns); rec["mapkf_rows"] = len(df)
        sp = mod_path("siglip", vid)
        if sp and os.path.exists(sp):
            a = np.load(sp, mmap_mode="r")
            rec["siglip_shape"] = str(a.shape); rec["siglip_dtype"] = str(a.dtype)
            rec["siglip_norm0"] = round(float(np.linalg.norm(np.asarray(a[0], dtype=np.float32))), 4)
            rec["siglip_nan"] = bool(~np.isfinite(np.asarray(a[:8], dtype=np.float32)).all())
        d = jload(mod_path("caption", vid) or "")
        if d:
            rec["cap_kf"] = len(d.get("keyframes", []))
            if d.get("keyframes"):
                rec["cap_item_keys"] = ",".join(d["keyframes"][0].keys())
        d = jload(mod_path("ocr", vid) or "")
        if d:
            rec["ocr_kf"] = len(d.get("keyframes", []))
            rec["ocr_nonempty"] = sum(1 for k in d.get("keyframes", []) if (k.get("text") or "").strip())
        d = jload(mod_path("asr_en", vid) or "")
        if d:
            rec["asr_segs"] = len(d.get("segments", []))
            rec["asr_has_en"] = bool(d.get("segments") and "text_en" in d["segments"][0])
        d = jload(mod_path("summary", vid) or "")
        if d:
            rec["sum_len"] = len(d.get("summary") or "")
        rep.append(rec)
    return pd.DataFrame(rep)

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 60)
schema_rep = audit_schema()
print(schema_rep.to_string(index=False))
schema_rep.to_csv(ART / "audit_schema.csv", index=False)

prob = []
for vid in VIDEO_IDS[:60]:
    sp, mp = mod_path("siglip", vid), mod_path("mapkf", vid)
    if not (sp and os.path.exists(sp) and mp and os.path.exists(mp)):
        continue
    a = np.load(sp, mmap_mode="r"); n_map = len(pd.read_csv(mp))
    if a.shape[1] != CFG["SIGLIP_DIM"]:
        prob.append((vid, "dim", a.shape))
    if a.shape[0] != n_map:
        prob.append((vid, "rowcount", (a.shape[0], n_map)))
print("\nP0 mismatch (siglip dim / rowcount vs map-keyframes):", prob[:10] or "NONE OK")

In [ ]:
# ============================================================
# CELL 7 — Canonical records: keyframes / caption / ocr / objects
#   canonical: video_id, keyframe_n, frame_idx, pts_time, start_time,
#              end_time, modality, language, text, source_id,
#              source_score, is_missing
# ============================================================
FP_RECORDS = sha(
    dir_fingerprint(PATHS["mapkf"], "*.csv"),
    dir_fingerprint(PATHS["caption"], "*.json"),
    dir_fingerprint(PATHS["ocr"], "*.json"),
    dir_fingerprint(PATHS["objects"], "*/*.json", limit=4000),
    CFG["LIMIT_VIDEOS"], CFG["OBJ_CONF_MIN"], CFG["OCR_CONF_MIN"],
)
print("records fingerprint:", FP_RECORDS)

def build_keyframe_records():
    kf_rows, cap_rows, ocr_rows, obj_rows = [], [], [], []
    for vi, vid in enumerate(VIDEO_IDS):
        if vi % 100 == 0:
            print(f"  [{vi}/{len(VIDEO_IDS)}] {vid}", flush=True); check_budget("records")
        kmap = None
        mp = mod_path("mapkf", vid)
        if mp and os.path.exists(mp):
            try:
                kmap = pd.read_csv(mp)
            except Exception as e:
                log_err("mapkf", vid, e)
        cap = jload(mod_path("caption", vid) or "", {}) or {}
        ocr = jload(mod_path("ocr", vid) or "", {}) or {}
        cap_by_n = {int(k["n"]): k for k in cap.get("keyframes", []) if k.get("n") is not None}
        ocr_by_n = {int(k["n"]): k for k in ocr.get("keyframes", []) if k.get("n") is not None}

        if kmap is None or len(kmap) == 0:
            ns = sorted(set(cap_by_n) | set(ocr_by_n))
            if not ns:
                continue
            kmap = pd.DataFrame([{"n": n,
                                  "frame_idx": (cap_by_n.get(n) or ocr_by_n.get(n) or {}).get("frame_idx", -1),
                                  "pts_time": (cap_by_n.get(n) or ocr_by_n.get(n) or {}).get("pts_time", float("nan")),
                                  "fps": (cap_by_n.get(n) or ocr_by_n.get(n) or {}).get("fps", float("nan"))}
                                 for n in ns])

        for _, r in kmap.iterrows():
            n = int(r["n"])
            fi = int(r["frame_idx"]) if pd.notna(r.get("frame_idx")) else -1
            pts = float(r["pts_time"]) if pd.notna(r.get("pts_time")) else float("nan")
            fps = float(r["fps"]) if pd.notna(r.get("fps")) else float("nan")
            kf_rows.append(dict(video_id=vid, keyframe_n=n, frame_idx=fi, pts_time=pts, fps=fps))

            c = cap_by_n.get(n)
            if c and (c.get("caption") or "").strip():
                cap_rows.append(dict(video_id=vid, keyframe_n=n, frame_idx=fi, pts_time=pts,
                                     modality="caption", language="en", text=c["caption"].strip(),
                                     source_id=f"caption:{vid}:{n}", source_score=1.0, is_missing=False))
            o = ocr_by_n.get(n)
            if o:
                kept = [d for d in (o.get("detections") or [])
                        if d.get("kept", True) and float(d.get("confidence", 0)) >= CFG["OCR_CONF_MIN"]]
                txt = " ".join((d.get("text") or "").strip() for d in kept).strip() or (o.get("text") or "").strip()
                if txt:
                    conf = float(np.mean([d.get("confidence", 0) for d in kept])) if kept else 0.0
                    ocr_rows.append(dict(video_id=vid, keyframe_n=n, frame_idx=fi, pts_time=pts,
                                         modality="ocr", language="vi", text=txt,
                                         source_id=f"ocr:{vid}:{n}", source_score=conf, is_missing=False))
        od = mod_path("objects", vid)
        if od and os.path.isdir(od):
            for p in sorted(glob.glob(os.path.join(od, "*.json"))):
                try:
                    n = int(Path(p).stem)
                except Exception:
                    continue
                d = jload(p, {}) or {}
                ents = d.get("detection_class_entities") or []
                scs = [float(x) for x in (d.get("detection_scores") or [])]
                best = {}
                for e, s in zip(ents, scs):
                    if s >= CFG["OBJ_CONF_MIN"]:
                        best[e] = max(best.get(e, 0.0), s)
                if best:
                    obj_rows.append(dict(video_id=vid, keyframe_n=n,
                                         labels=list(best.keys()),
                                         scores=[round(v, 4) for v in best.values()],
                                         count=int(sum(1 for s in scs if s >= CFG["OBJ_CONF_MIN"]))))
    return (pd.DataFrame(kf_rows), pd.DataFrame(cap_rows),
            pd.DataFrame(ocr_rows), pd.DataFrame(obj_rows))

if stage_done("records_kf", FP_RECORDS) and load_parquet("keyframes") is not None:
    print("SKIP records_kf (fingerprint unchanged)")
    KF, CAP, OCR, OBJ = (load_parquet("keyframes"), load_parquet("caption"),
                         load_parquet("ocr"), load_parquet("objects"))
else:
    with Timer("build canonical keyframe records"):
        KF, CAP, OCR, OBJ = build_keyframe_records()
    for nm, df in [("keyframes", KF), ("caption", CAP), ("ocr", OCR), ("objects", OBJ)]:
        save_parquet(df, nm)
    mark_stage("records_kf", FP_RECORDS, keyframes=len(KF), caption=len(CAP), ocr=len(OCR), objects=len(OBJ))

print(f"\nKF={len(KF):,}  CAP={len(CAP):,}  OCR={len(OCR):,}  OBJ={len(OBJ):,}")
display(KF.head(3)); display(CAP.head(2)); display(OCR.head(2))

In [ ]:
# ============================================================
# CELL 8 — Canonical records: transcript chunk (vi/en) + summary/metadata (video-level)
# ============================================================
FP_TEXT = sha(dir_fingerprint(PATHS["asr_en"], "**/*.json"),
              dir_fingerprint(PATHS["asr_vi"], "**/*.json"),
              dir_fingerprint(PATHS["summary"], "*.json"),
              dir_fingerprint(PATHS["media"], "*.json"),
              CFG["TRANSCRIPT_CHUNK_SEC"], CFG["TRANSCRIPT_OVERLAP_SEC"], CFG["LIMIT_VIDEOS"])

def chunk_segments(segs, win, ov):
    """Gộp segment ASR thành chunk ~win giây, overlap ov giây; giữ nguyên timestamp gốc."""
    out, i = [], 0
    while i < len(segs):
        j, t0 = i, float(segs[i].get("start", 0.0))
        while j < len(segs) and float(segs[j].get("end", t0)) - t0 < win:
            j += 1
        j = max(j, i + 1)
        grp = segs[i:j]
        out.append(dict(start=float(grp[0].get("start", 0.0)), end=float(grp[-1].get("end", 0.0)),
                        vi=" ".join((g.get("text") or "").strip() for g in grp).strip(),
                        en=" ".join((g.get("text_en") or "").strip() for g in grp).strip()))
        if j >= len(segs):
            break
        back, k = float(grp[-1].get("end", 0.0)) - ov, j
        while k > i + 1 and float(segs[k - 1].get("start", 0.0)) > back:
            k -= 1
        i = max(k, i + 1)
    return out

FPS_BY_VID = {}
if len(KF):
    _t = KF.dropna(subset=["fps"]).groupby("video_id")["fps"].first()
    FPS_BY_VID = {k: float(v) for k, v in _t.items()}

def build_text_records():
    tr_rows, sm_rows = [], []
    for vi_, vid in enumerate(VIDEO_IDS):
        if vi_ % 150 == 0:
            print(f"  [{vi_}/{len(VIDEO_IDS)}] {vid}", flush=True); check_budget("text-records")
        d = jload(mod_path("asr_en", vid) or "") or jload(mod_path("asr_vi", vid) or "") or {}
        segs = d.get("segments") or []
        fps_v = FPS_BY_VID.get(vid)
        for ci, ch in enumerate(chunk_segments(segs, CFG["TRANSCRIPT_CHUNK_SEC"], CFG["TRANSCRIPT_OVERLAP_SEC"])):
            mid = (ch["start"] + ch["end"]) / 2.0
            tr_rows.append(dict(video_id=vid, chunk_id=ci, start_time=ch["start"], end_time=ch["end"],
                                mid_time=mid, frame_idx=int(mid * fps_v) if fps_v else -1,
                                modality="transcript", text_vi=ch["vi"], text_en=ch["en"],
                                source_id=f"asr:{vid}:{ci}", source_score=1.0,
                                is_missing=not bool(ch["vi"] or ch["en"])))
        su = jload(mod_path("summary", vid) or "", {}) or {}
        me = jload(mod_path("media", vid) or "", {}) or {}
        meta_txt = " | ".join(str(me.get(k, "")) for k in ("title", "author", "keywords", "publish_date") if me.get(k))
        if not meta_txt and me.get("description"):
            meta_txt = str(me["description"])[:2000]
        sm_rows.append(dict(video_id=vid, summary_en=(su.get("summary") or "").strip(),
                            meta_text=meta_txt.strip(),
                            length_sec=float(me.get("length") or 0) or None,
                            is_missing=not bool((su.get("summary") or "").strip() or meta_txt.strip())))
    return pd.DataFrame(tr_rows), pd.DataFrame(sm_rows)

if stage_done("records_text", FP_TEXT) and load_parquet("transcript") is not None:
    print("SKIP records_text"); TR, SM = load_parquet("transcript"), load_parquet("summary")
else:
    with Timer("build transcript/summary records"):
        TR, SM = build_text_records()
    save_parquet(TR, "transcript"); save_parquet(SM, "summary")
    mark_stage("records_text", FP_TEXT, transcript=len(TR), summary=len(SM))

print(f"TR={len(TR):,} chunks | SM={len(SM):,} videos "
      f"| summary rỗng={int(SM.summary_en.eq('').sum())} | meta rỗng={int(SM.meta_text.eq('').sum())}")
display(TR.head(2)); display(SM.head(2))

In [ ]:
# ============================================================
# CELL 9 — Index 1: SigLIP2 visual FAISS (IndexFlatIP; cosine trên vector đã L2-norm)
#   Mỗi embedding space MỘT index riêng — không trộn SigLIP2 / BGE / CLIP.
# ============================================================
FP_SIGLIP = sha(dir_fingerprint(PATHS["siglip"], "*.npy"), CFG["LIMIT_VIDEOS"], CFG["SIGLIP_DIM"])
KF_BY_VID = {v: g.sort_values("keyframe_n") for v, g in KF.groupby("video_id")} if len(KF) else {}

def build_visual_index(kind="siglip", dim=None):
    dim = dim or (CFG["SIGLIP_DIM"] if kind == "siglip" else 512)
    index = faiss.IndexFlatIP(dim)
    rowmap, bad, buf = [], [], []
    BUF_MAX = 200_000
    for vi_, vid in enumerate(VIDEO_IDS):
        p = mod_path(kind, vid)
        if not (p and os.path.exists(p)):
            continue
        try:
            a = np.load(p).astype(np.float32)
        except Exception as e:
            log_err(kind, vid, e); continue
        if a.ndim != 2 or a.shape[1] != dim:
            bad.append((vid, "dim", str(a.shape))); continue
        if not np.isfinite(a).all():
            bad.append((vid, "naninf", int((~np.isfinite(a)).any(1).sum())))
            a = np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)
        nrm = np.linalg.norm(a, axis=1, keepdims=True); nrm[nrm == 0] = 1.0
        a = a / nrm
        kfv = KF_BY_VID.get(vid)
        if kfv is not None and len(kfv) != len(a):
            bad.append((vid, "rowcount", (len(a), len(kfv))))
        m = min(len(a), len(kfv)) if kfv is not None else len(a)
        if kfv is not None:
            sub = kfv.iloc[:m]
            rowmap += list(zip(sub.video_id, sub.keyframe_n, sub.frame_idx, sub.pts_time))
        else:
            rowmap += [(vid, i + 1, -1, float("nan")) for i in range(m)]
        buf.append(a[:m])
        if sum(x.shape[0] for x in buf) >= BUF_MAX:
            index.add(np.concatenate(buf)); buf = []
        if vi_ % 150 == 0:
            print(f"  [{vi_}/{len(VIDEO_IDS)}] ntotal≈{index.ntotal + sum(x.shape[0] for x in buf):,}", flush=True)
            check_budget("visual-index")
    if buf:
        index.add(np.concatenate(buf))
    rm = pd.DataFrame(rowmap, columns=["video_id", "keyframe_n", "frame_idx", "pts_time"])
    return index, rm, bad

if stage_done("index_siglip", FP_SIGLIP) and (ART / "index" / "siglip.faiss").exists():
    print("SKIP index_siglip")
    SIG_INDEX = faiss.read_index(str(ART / "index" / "siglip.faiss"))
    SIG_ROWMAP = pd.read_parquet(ART / "index" / "siglip_rowmap.parquet")
else:
    with Timer("build SigLIP2 FAISS index"):
        SIG_INDEX, SIG_ROWMAP, bad = build_visual_index("siglip")
    assert SIG_INDEX.ntotal == len(SIG_ROWMAP), (SIG_INDEX.ntotal, len(SIG_ROWMAP))
    faiss.write_index(SIG_INDEX, str(ART / "index" / "siglip.faiss"))
    SIG_ROWMAP.to_parquet(ART / "index" / "siglip_rowmap.parquet", index=False)
    jdump(bad, ART / "index" / "siglip_issues.json")
    mark_stage("index_siglip", FP_SIGLIP, ntotal=int(SIG_INDEX.ntotal), issues=len(bad))
    print("issues:", bad[:8] or "NONE OK")

print(f"SigLIP2 index: ntotal={SIG_INDEX.ntotal:,} dim={SIG_INDEX.d} | rowmap={len(SIG_ROWMAP):,}")
q = SIG_INDEX.reconstruct(0).reshape(1, -1)
D, I = SIG_INDEX.search(q, 3)
print("smoke self-query:", I[0][:3], np.round(D[0][:3], 4),
      "->", "OK" if I[0][0] == 0 and D[0][0] > 0.99 else "FAIL")

In [ ]:
# ============================================================
# CELL 10 — BGE-M3 encoder (FP16 trên T4: load -> encode theo shard -> unload)
# ============================================================
from transformers import AutoTokenizer, AutoModel

_BGE = {"tok": None, "model": None}

def bge_load():
    if _BGE["model"] is not None:
        return
    name = CFG["LOCAL_MODEL_DIR"] and os.path.join(CFG["LOCAL_MODEL_DIR"], "bge-m3")
    name = name if (name and os.path.isdir(name)) else CFG["BGE_M3"]
    kw = dict(revision=CFG["BGE_M3_REV"]) if CFG["BGE_M3_REV"] else {}
    tok = AutoTokenizer.from_pretrained(name, **kw)
    dt = torch.float16 if CFG["BGE_FP16"] else torch.float32
    try:                                 # transformers 5.x
        mdl = AutoModel.from_pretrained(name, dtype=dt, **kw)
    except TypeError:                    # transformers 4.x
        mdl = AutoModel.from_pretrained(name, torch_dtype=dt, **kw)
    mdl = mdl.to("cuda:0" if torch.cuda.is_available() else "cpu").eval()
    _BGE.update(tok=tok, model=mdl, name=name)
    print("BGE-M3 loaded:", name, "| dtype:", next(mdl.parameters()).dtype)

def bge_unload():
    _BGE["model"] = None; _BGE["tok"] = None; free_gpu(); print("BGE-M3 unloaded")

@torch.inference_mode()
def bge_encode(texts, batch=None, maxlen=None, desc=""):
    """CLS pooling + L2 normalize (dense của bge-m3). Tự giảm batch khi OOM."""
    bge_load()
    tok, mdl = _BGE["tok"], _BGE["model"]
    dev = next(mdl.parameters()).device
    batch = batch or CFG["BGE_BATCH"]; maxlen = maxlen or CFG["BGE_MAXLEN"]
    out = np.zeros((len(texts), CFG["BGE_DIM"]), dtype=np.float32)
    i, step = 0, 0
    while i < len(texts):
        b = batch
        while True:
            try:
                chunk = [t if (t or "").strip() else " " for t in texts[i:i + b]]
                enc = tok(chunk, padding=True, truncation=True, max_length=maxlen, return_tensors="pt").to(dev)
                o = mdl(**enc)
                hs = getattr(o, "last_hidden_state", None)
                h = hs[:, 0] if hs is not None else o.pooler_output
                h = torch.nn.functional.normalize(h.float(), dim=-1)
                out[i:i + len(chunk)] = h.cpu().numpy()
                break
            except torch.cuda.OutOfMemoryError:
                free_gpu(); b = max(1, b // 2); print(f"    OOM -> batch={b}")
                if b == 1:
                    maxlen = max(128, maxlen // 2); print(f"    OOM at batch=1 -> maxlen={maxlen}")
        i += b; step += 1
        if desc and step % 100 == 0:
            print(f"    {desc} {i}/{len(texts)}", flush=True); check_budget("bge-encode")
    assert np.isfinite(out).all(), "BGE output có NaN/Inf"
    return out

_v = bge_encode(["một người mặc áo đỏ phát biểu", "a person in a red shirt speaking", "công thức nấu ăn"])
print("smoke BGE: norms", np.round(np.linalg.norm(_v, axis=1), 4),
      "| cos(vi,en)", round(float(_v[0] @ _v[1]), 3),
      "| cos(vi,unrelated)", round(float(_v[0] @ _v[2]), 3))
assert float(_v[0] @ _v[1]) > float(_v[0] @ _v[2]), "FP16 sanity FAIL - xem lại dtype"

In [ ]:
# ============================================================
# CELL 11 — Index 2/3/4: BGE-M3 dense (caption / transcript-chunk / summary+metadata)
#   shard checkpoint -> resume; skip khi fingerprint không đổi
# ============================================================
def build_dense_index(name, texts, meta_df, fp):
    idx_p = ART / "index" / f"{name}.faiss"
    map_p = ART / "index" / f"{name}_rowmap.parquet"
    if stage_done(f"index_{name}", fp) and idx_p.exists():
        print(f"SKIP index_{name}")
        return faiss.read_index(str(idx_p)), pd.read_parquet(map_p)

    shard_dir = ART / "ckpt" / f"emb_{name}"; shard_dir.mkdir(parents=True, exist_ok=True)
    S = CFG["SHARD_SIZE"]
    n_shard = math.ceil(len(texts) / S) if len(texts) else 0
    for s in range(n_shard):
        sp = shard_dir / f"{fp}_{s:04d}.npy"
        if sp.exists():
            continue
        check_budget(f"dense-{name}")
        with Timer(f"{name} shard {s+1}/{n_shard}"):
            np.save(sp, bge_encode(texts[s * S:(s + 1) * S], desc=f"{name}#{s}"))
    index = faiss.IndexFlatIP(CFG["BGE_DIM"])
    for s in range(n_shard):
        index.add(np.load(shard_dir / f"{fp}_{s:04d}.npy").astype(np.float32))
    assert index.ntotal == len(texts), (index.ntotal, len(texts))
    faiss.write_index(index, str(idx_p))
    meta_df.reset_index(drop=True).to_parquet(map_p, index=False)
    mark_stage(f"index_{name}", fp, ntotal=int(index.ntotal))
    print(f"  -> {name}.faiss ntotal={index.ntotal:,}")
    return index, meta_df.reset_index(drop=True)

cap_meta = CAP[["video_id", "keyframe_n", "frame_idx", "pts_time", "source_id"]].copy()
CAP_INDEX, CAP_MAP = build_dense_index("cap_dense", CAP.text.tolist(), cap_meta,
                                       sha(FP_RECORDS, "cap", CFG["BGE_M3"], CFG["BGE_MAXLEN"]))

tr_text = [(e or v or " ") for e, v in zip(TR.text_en, TR.text_vi)]
tr_meta = TR[["video_id", "chunk_id", "start_time", "end_time", "mid_time", "frame_idx", "source_id"]].copy()
TR_INDEX, TR_MAP = build_dense_index("tr_dense", tr_text, tr_meta,
                                     sha(FP_TEXT, "tr", CFG["BGE_M3"], CFG["BGE_MAXLEN"]))

sm_text = [(f"{s} {m}").strip() or " " for s, m in zip(SM.summary_en, SM.meta_text)]
sm_meta = SM[["video_id"]].copy()
SM_INDEX, SM_MAP = build_dense_index("sm_dense", sm_text, sm_meta, sha(FP_TEXT, "sm", CFG["BGE_M3"]))

bge_unload()
print(f"\ndense: cap={CAP_INDEX.ntotal:,}  transcript={TR_INDEX.ntotal:,}  summary={SM_INDEX.ntotal:,}")

In [ ]:
# ============================================================
# CELL 12 — Index 5/6: BM25 multi-field (thuần Python, không cần network)
#   Multi-field thay vì nhân đôi document: field "raw" (có dấu) + "nodia" (bỏ dấu), weight riêng.
# ============================================================
class BM25MultiField:
    """BM25 Okapi trên nhiều field; score = Σ_field weight * bm25_field."""
    def __init__(self, k1=1.2, b=0.75):
        self.k1, self.b = k1, b
        self.fields = {}
        self.n_doc = 0

    def add_field(self, field, docs_tokens, weight=1.0):
        postings, df, dl = {}, {}, np.zeros(len(docs_tokens), dtype=np.int32)
        for i, toks in enumerate(docs_tokens):
            dl[i] = len(toks)
            tf = {}
            for t in toks:
                tf[t] = tf.get(t, 0) + 1
            for t, c in tf.items():
                postings.setdefault(t, []).append((i, c))
                df[t] = df.get(t, 0) + 1
        self.fields[field] = dict(postings={t: np.array(v, dtype=np.int32) for t, v in postings.items()},
                                  df=df, dl=dl, avgdl=float(dl.mean()) if len(dl) else 1.0,
                                  N=len(docs_tokens), weight=float(weight))
        self.n_doc = max(self.n_doc, len(docs_tokens))
        return self

    def _idf(self, F, t):
        n = F["df"].get(t, 0)
        return math.log(1 + (F["N"] - n + 0.5) / (n + 0.5))

    def search(self, query_by_field, topk=200):
        scores = {}
        for field, toks in query_by_field.items():
            F = self.fields.get(field)
            if not F or not toks:
                continue
            w, k1, b = F["weight"], self.k1, self.b
            avgdl = max(F["avgdl"], 1e-6)
            for t in set(toks):
                pl = F["postings"].get(t)
                if pl is None:
                    continue
                idf = self._idf(F, t)
                docs, tf = pl[:, 0], pl[:, 1].astype(np.float32)
                dl = F["dl"][docs].astype(np.float32)
                s = w * idf * (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * dl / avgdl))
                for d, v in zip(docs, s):
                    scores[int(d)] = scores.get(int(d), 0.0) + float(v)
        if not scores:
            return np.zeros(0, np.int64), np.zeros(0, np.float32)
        items = sorted(scores.items(), key=lambda x: -x[1])[:topk]
        return (np.array([i for i, _ in items], np.int64),
                np.array([s for _, s in items], np.float32))

def bm25_dump(bm):
    """Serialize thành dict thuần (numpy + builtin) — KHÔNG pickle custom class,
    tránh phụ thuộc __main__ giữa các notebook."""
    return dict(k1=bm.k1, b=bm.b, n_doc=bm.n_doc, fields=bm.fields)

def bm25_load(d):
    bm = BM25MultiField(k1=d["k1"], b=d["b"])
    bm.fields = d["fields"]; bm.n_doc = d["n_doc"]
    return bm

def build_bm25(name, texts, meta_df, fp, weights=(1.0, 0.7)):
    p = ART / "index" / f"bm25_{name}.pkl"
    if stage_done(f"bm25_{name}", fp) and p.exists():
        print(f"SKIP bm25_{name}")
        with open(p, "rb") as f:
            d = pickle.load(f)
        return dict(bm25=bm25_load(d["bm25"]), meta=d["meta"])
    with Timer(f"bm25 {name} ({len(texts):,} docs)"):
        bm = BM25MultiField()
        bm.add_field("raw", [tokenize(t) for t in texts], weight=weights[0])
        bm.add_field("nodia", [tokenize_nodia(t) for t in texts], weight=weights[1])
    obj = dict(bm25=bm25_dump(bm), meta=meta_df.reset_index(drop=True))
    with open(p, "wb") as f:
        pickle.dump(obj, f, protocol=4)
    mark_stage(f"bm25_{name}", fp, ndoc=len(texts))
    print(f"  -> bm25_{name}.pkl  {p.stat().st_size/1e6:.1f} MB")
    return dict(bm25=bm, meta=obj["meta"])

BM = {}
BM["ocr"] = build_bm25("ocr", OCR.text.tolist(),
                       OCR[["video_id", "keyframe_n", "frame_idx", "pts_time", "source_score"]],
                       sha(FP_RECORDS, "ocr"), weights=(1.0, 0.8))
BM["caption"] = build_bm25("caption", CAP.text.tolist(), cap_meta, sha(FP_RECORDS, "capbm"))
BM["transcript"] = build_bm25("transcript",
                              [f"{v} \n {e}" for v, e in zip(TR.text_vi, TR.text_en)],
                              tr_meta, sha(FP_TEXT, "trbm"), weights=(1.0, 0.8))
BM["summary"] = build_bm25("summary", sm_text, sm_meta, sha(FP_TEXT, "smbm"))
print("\nBM25 indices:", list(BM))

qi, qs = BM["transcript"]["bm25"].search({"raw": tokenize("sụt lún đồng bằng sông cửu long"),
                                          "nodia": tokenize_nodia("sut lun dong bang song cuu long")}, topk=3)
print("smoke BM25 transcript:", [(BM["transcript"]["meta"].video_id[i], round(float(s), 2)) for i, s in zip(qi, qs)])

In [ ]:
# ============================================================
# CELL 13 — Index 7: object inverted index (threshold + soft-boost)
# ============================================================
FP_OBJ = sha(FP_RECORDS, "objidx", CFG["OBJ_CONF_MIN"])
obj_p = ART / "index" / "objects_inverted.pkl"

if stage_done("index_obj", FP_OBJ) and obj_p.exists():
    with open(obj_p, "rb") as f:
        OBJ_IDX = pickle.load(f)
    print("SKIP index_obj")
else:
    with Timer("build object inverted index"):
        inv = {}
        for i, r in enumerate(OBJ.itertuples(index=False)):
            for lb, sc in zip(r.labels, r.scores):
                inv.setdefault(str(lb).lower(), []).append((i, float(sc)))
        inv = {k: np.array(v, dtype=np.float32) for k, v in inv.items()}
        OBJ_IDX = dict(inv=inv, meta=OBJ[["video_id", "keyframe_n", "count"]].reset_index(drop=True))
    with open(obj_p, "wb") as f:
        pickle.dump(OBJ_IDX, f, protocol=4)
    mark_stage("index_obj", FP_OBJ, n_label=len(OBJ_IDX["inv"]))

print("object labels:", len(OBJ_IDX["inv"]))
print("top labels   :", sorted(((len(v), k) for k, v in OBJ_IDX["inv"].items()), reverse=True)[:12])

In [ ]:
# ============================================================
# CELL 14 — Integrity check: load lại từ disk + smoke query
# ============================================================
def integrity_check():
    rep = []
    def ck(name, cond, detail=""):
        rep.append(dict(check=name, status="PASS" if cond else "FAIL", detail=str(detail)))
    for f in ["siglip.faiss", "cap_dense.faiss", "tr_dense.faiss", "sm_dense.faiss",
              "bm25_ocr.pkl", "bm25_caption.pkl", "bm25_transcript.pkl", "bm25_summary.pkl",
              "objects_inverted.pkl"]:
        ck(f"exists:{f}", (ART / "index" / f).exists())
    ix = faiss.read_index(str(ART / "index" / "siglip.faiss"))
    rm = pd.read_parquet(ART / "index" / "siglip_rowmap.parquet")
    ck("siglip ntotal==rowmap", ix.ntotal == len(rm), (ix.ntotal, len(rm)))
    ck("siglip dim", ix.d == CFG["SIGLIP_DIM"], ix.d)
    ck("cap ntotal==CAP", faiss.read_index(str(ART / "index" / "cap_dense.faiss")).ntotal == len(CAP))
    ck("tr ntotal==TR", faiss.read_index(str(ART / "index" / "tr_dense.faiss")).ntotal == len(TR))
    ck("sm ntotal==SM", faiss.read_index(str(ART / "index" / "sm_dense.faiss")).ntotal == len(SM))
    bad_fi = int((KF.frame_idx < 0).sum())
    ck("frame_idx valid >=99%", bad_fi <= 0.01 * max(len(KF), 1), f"{bad_fi}/{len(KF)} âm")
    ck("fps present >95%", float(KF.fps.notna().mean()) > 0.95, round(float(KF.fps.notna().mean()), 3))
    v = ix.reconstruct(min(5, ix.ntotal - 1))
    ck("siglip L2-normalized", abs(float(np.linalg.norm(v)) - 1.0) < 1e-2, float(np.linalg.norm(v)))
    ck("error ledger nhỏ", len(ERRORS) < 0.02 * max(len(VIDEO_IDS), 1), f"{len(ERRORS)} lỗi")
    return pd.DataFrame(rep)

report = integrity_check()
print(report.to_string(index=False))
report.to_csv(ART / "integrity_report.csv", index=False)
jdump(ERRORS, ART / "error_ledger.json")
FAILS = report[report.status == "FAIL"]
print("\n", "ALL PASS" if FAILS.empty else f"{len(FAILS)} CHECK FAIL - xử lý trước khi sang NB02")

In [ ]:
# ============================================================
# CELL 15 — Manifest + bàn giao NB01 -> NB02
# ============================================================
MANIFEST = dict(
    notebook="01_build_indices_local",
    created=time.strftime("%Y-%m-%d %H:%M:%S"),
    seed=CFG["SEED"],
    roots=dict(dataset=DATASET_ROOT, feature=FEATURE_ROOT, query=QUERY_ROOT),
    modality_paths=PATHS,
    models=dict(bge_m3=CFG["BGE_M3"], bge_m3_revision=CFG["BGE_M3_REV"],
                siglip2="google/siglip2-giant-opt-patch16-384 (keyframe embedding đã extract sẵn)",
                bge_dim=CFG["BGE_DIM"], siglip_dim=CFG["SIGLIP_DIM"]),
    params={k: CFG[k] for k in ("LIMIT_VIDEOS", "BGE_BATCH", "BGE_MAXLEN", "BGE_FP16", "SHARD_SIZE",
                                "TRANSCRIPT_CHUNK_SEC", "TRANSCRIPT_OVERLAP_SEC",
                                "OBJ_CONF_MIN", "OCR_CONF_MIN")},
    fingerprints=dict(records=FP_RECORDS, text=FP_TEXT, siglip=FP_SIGLIP, objects=FP_OBJ),
    counts=dict(videos=len(VIDEO_IDS), keyframes=len(KF), caption=len(CAP), ocr=len(OCR),
                objects=len(OBJ), transcript_chunks=len(TR), summaries=len(SM),
                siglip_vectors=int(SIG_INDEX.ntotal)),
    artifacts=dict(
        records=[f"records/{n}.parquet" for n in
                 ("keyframes", "caption", "ocr", "objects", "transcript", "summary")],
        index=[f"index/{n}" for n in ("siglip.faiss", "siglip_rowmap.parquet",
                                      "cap_dense.faiss", "cap_dense_rowmap.parquet",
                                      "tr_dense.faiss", "tr_dense_rowmap.parquet",
                                      "sm_dense.faiss", "sm_dense_rowmap.parquet",
                                      "bm25_ocr.pkl", "bm25_caption.pkl", "bm25_transcript.pkl",
                                      "bm25_summary.pkl", "objects_inverted.pkl")],
        reports=["audit_coverage.csv", "audit_schema.csv", "integrity_report.csv", "error_ledger.json"],
    ),
    integrity_all_pass=bool(FAILS.empty),
    runtime_sec=round(time.time() - T0, 1),
    stage_state=STAGE,
)
jdump(MANIFEST, ART / "manifest.json")
print(json.dumps({k: MANIFEST[k] for k in ("counts", "integrity_all_pass", "runtime_sec")},
                 indent=1, ensure_ascii=False))
print(f"""
=== BÀN GIAO NB01 -> NB02 ===
Artifact dir: {ART}
Bước tiếp:
  1. Save & Version notebook này (Save output) -> Kaggle sinh dataset output.
  2. Trong NB02 đặt CFG["ART_INPUT"] = "/kaggle/input/<slug-output-nb01>/artifacts".
  3. NB02 chỉ load artifact, KHÔNG build lại index (fingerprint đã ghi trong manifest.json).
Runtime: {(time.time()-T0)/60:.1f} phút | budget còn {budget_left()/3600:.2f} h
""")